In [1]:
import torch.nn as nn
from torch import Tensor
from torchsummary import summary

class Classificator(nn.Module):

    def __init__(
        self,
        n_classes: int,
    ) -> None:
        super().__init__()
        norm_layer = nn.BatchNorm2d
        self.conv0 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn0 = norm_layer(64)
        self.relu = nn.ReLU()
            
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        
        self.conv1 = nn.Conv2d(64, 16, kernel_size=1, padding='same', bias=False)
        self.bn1 = norm_layer(16)

        self.conv2 = nn.Conv2d(16, 16, kernel_size=3, padding='same', bias=False)
        self.bn2 = norm_layer(16)
        
        self.conv3 = nn.Conv2d(16, 64, kernel_size=1, padding='same', bias=False)
        self.bn3 = norm_layer(64)
        
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(64, n_classes)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x: Tensor) -> Tensor:
        out = self.conv0(x)
        out = self.bn0(out)
        out = self.relu(out)

        out = self.maxpool(out)
        
        identity = out

        out = self.conv1(out)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out)
        out = self.bn3(out)
        
        out += identity
        out = self.relu(out)

        out = self.avgpool(out)
        out = self.flatten(out)
        out = self.fc(out)
        out = self.softmax(out)

        return out

model = Classificator(10)
print("Количество параметров:", sum(p.numel() for p in model.parameters()))

summary(model, input_size=(3, 224, 224),  device = 'cpu')

Количество параметров: 14730
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 64, 112, 112]           9,408
       BatchNorm2d-2         [-1, 64, 112, 112]             128
              ReLU-3         [-1, 64, 112, 112]               0
         MaxPool2d-4           [-1, 64, 56, 56]               0
            Conv2d-5           [-1, 16, 56, 56]           1,024
       BatchNorm2d-6           [-1, 16, 56, 56]              32
              ReLU-7           [-1, 16, 56, 56]               0
            Conv2d-8           [-1, 16, 56, 56]           2,304
       BatchNorm2d-9           [-1, 16, 56, 56]              32
             ReLU-10           [-1, 16, 56, 56]               0
           Conv2d-11           [-1, 64, 56, 56]           1,024
      BatchNorm2d-12           [-1, 64, 56, 56]             128
             ReLU-13           [-1, 64, 56, 56]               0
AdaptiveAv